# 4.3 Neural Network Implementation
### Software Automation: Machine Learning & AI


**Learning intentions**
- Apply a neural network model using OOP to make predictions on a dataset
- Evaluate model accuracy and identify areas for improvement

---
**How to use this notebook**
Cells marked 🔨 **TODO** contain gaps for you to complete. Read the explanation above each one before writing your code.
Cells with complete code are provided — read them carefully before moving on.


---
## Part 1 — OOP Design

Before writing any code, let's plan the class structure.

| Attribute | Type | Purpose |
|-----------|------|---------|
| `input_size` | `int` | Number of input features |
| `hidden_size` | `int` | Number of nodes in the hidden layer |
| `output_size` | `int` | Number of output nodes (1 for binary classification) |
| `learning_rate` | `float` | Step size for gradient descent weight updates |
| `W1`, `b1` | arrays | Weights and biases for the input → hidden connection |
| `W2`, `b2` | arrays | Weights and biases for the hidden → output connection |
| `loss_history` | list | Tracks training loss after each epoch |

| Method | Purpose |
|--------|---------|
| `__init__` | Set architecture, initialise weights randomly |
| `sigmoid` | Activation function applied at each layer |
| `sigmoid_derivative` | Derivative of sigmoid — needed for backpropagation |
| `forward` | Compute a prediction by passing input through all layers |
| `backward` | Compute gradients and update weights to reduce loss |
| `train` | Run forward + backward for every epoch over the dataset |
| `predict` | Return class labels (0 or 1) from the trained model |

The network architecture looks like this:

```
Input layer        Hidden layer       Output layer
(input_size)  →→→  (hidden_size)  →→→  (output_size)
    X             W1, b1, sigmoid      W2, b2, sigmoid
```


### Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fix the random seed so results are reproducible
np.random.seed(42)


---
### `__init__` — Set up the network

When we create a `NeuralNetwork` object we need to:
1. Store the architecture parameters
2. **Randomly initialise** the weight matrices — this is important because if all weights start at zero, every node computes the same gradient and the network never learns distinct features
3. Initialise biases to zero (safe — only *weights* need random starts to break symmetry)

`W1` connects the input layer to the hidden layer, so its shape is `(input_size, hidden_size)`.
`W2` connects the hidden layer to the output layer, so its shape is `(hidden_size, output_size)`.

We scale random values by `0.01` to keep initial weights small — large starting weights cause unstable gradients early in training.


In [4]:
class NeuralNetwork:

    def __init__(self, input_size, hidden_size, output_size, learning_rate=1.0):
        """
        Initialise the network architecture and weights.

        Parameters
        ----------
        input_size    : number of input features
        hidden_size   : number of nodes in the hidden layer
        output_size   : number of output nodes
        learning_rate : step size for gradient descent
        """
        
        self.input_size    = input_size
        self.hidden_size   = hidden_size
        self.output_size   = output_size
        self.learning_rate = learning_rate

        # ── Weight matrices (randomly initialised, scaled small) ──────────
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))

        # 🔨 TODO: Initialise W2 and b2 using the same pattern as W1 and b1 above.
        #          W2 connects the hidden layer to the output layer.
        self.W2 = np.random.randn(input_size, hidden_size) * 0.01   # replace None
        self.b2 = np.zeros((1, hidden_size))   # replace None

        # ── Training history ──────────────────────────────────────────────
        self.loss_history = []


    def sigmoid(self, z):
        """Apply the sigmoid activation function element-wise."""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, s):
        """
        Derivative of sigmoid given the sigmoid OUTPUT s (not the raw input z).
        Used during backpropagation.
        """
        return s * (1 - s)


---
### `sigmoid` and `sigmoid_derivative` — Activation functions

The **sigmoid** function squashes any value into the range (0, 1).
This lets us interpret the network's output as a probability.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The **derivative** of sigmoid is used during backpropagation to measure how sensitive each node's output is to changes in its input.
If we already have the sigmoid output `s`, the derivative simplifies neatly to:

$$\sigma'(z) = s \cdot (1 - s)$$

Both methods are provided below — read them carefully before continuing.


In [3]:
    def sigmoid(self, z):
        """Apply the sigmoid activation function element-wise."""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, s):
        """
        Derivative of sigmoid given the sigmoid OUTPUT s (not the raw input z).
        Used during backpropagation.
        """
        return s * (1 - s)


---
### `forward` — The forward pass

The forward pass computes the network's prediction by flowing data from left to right through the layers.

**Step 1 — Input → Hidden layer**
```
hidden_input  = X @ W1 + b1           # weighted sum of inputs
hidden_output = sigmoid(hidden_input) # apply activation
```

**Step 2 — Hidden → Output layer**
```
output_input  = hidden_output @ W2 + b2
output        = sigmoid(output_input)
```

The `@` operator is Python's matrix multiplication symbol.

> **Why store intermediate values?**
> `hidden_input`, `hidden_output`, and `output_input` are stored as instance attributes
> because the backward pass needs them to compute gradients.


In [12]:
    def forward(self, X):
        """
        Compute a prediction for input X.

        Parameters
        ----------
        X : array of shape (n_samples, input_size)

        Returns
        -------
        output : array of shape (n_samples, output_size)
        """
        # ── Step 1: Input → Hidden ─────────────────────────────────────────
        # 🔨 TODO: Compute hidden_input (weighted sum) and hidden_output (after activation).
        #          Store both as self.hidden_input and self.hidden_output.
        self.hidden_input  = None   # X @ ? + ?
        self.hidden_output = None   # sigmoid( ? )

        # ── Step 2: Hidden → Output ───────────────────────────────────────
        self.output_input = self.hidden_output @ self.W2 + self.b2
        output            = self.sigmoid(self.output_input)

        return output


---
### `backward` — Backpropagation

Backpropagation calculates how much each weight contributed to the prediction error,
then nudges every weight in the direction that reduces that error.

We use **mean squared error (MSE)** loss:

$$L = \frac{1}{2n} \sum (\hat{y} - y)^2$$

The factor of ½ is a convenience that cancels with the exponent when differentiating.
The division by `n` (number of samples) **averages** the gradients across the batch.

**Gradient flow (output → hidden → input)**

| Step | Formula |
|------|---------|
| Output error | `delta_output = (output − y) * sigmoid_derivative(output)` |
| Gradient for W2 | `dW2 = hidden_output.T @ delta_output / n` |
| Gradient for b2 | `db2 = sum(delta_output, axis=0) / n` |
| Propagate error back | `delta_hidden = (delta_output @ W2.T) * sigmoid_derivative(hidden_output)` |
| Gradient for W1 | `dW1 = X.T @ delta_hidden / n` |
| Gradient for b1 | `db1 = sum(delta_hidden, axis=0) / n` |

After computing all gradients, **subtract** them (scaled by the learning rate) from each weight.
> Gradients point uphill (toward higher loss). Subtracting moves us downhill toward lower loss.


In [16]:
    def backward(self, X, y, output):
        """
        Compute gradients and update all weights and biases.

        Parameters
        ----------
        X      : original input array, shape (n_samples, input_size)
        y      : true labels, shape (n_samples, 1)
        output : network prediction from forward(), shape (n_samples, output_size)
        """
        n = X.shape[0]   # number of samples — used to average gradients across the batch

        # ── Output layer gradients ─────────────────────────────────────────
        delta_output = (output - y) * self.sigmoid_derivative(output)
        dW2 = self.hidden_output.T @ delta_output / n
        db2 = np.sum(delta_output, axis=0, keepdims=True) / n

        # ── Hidden layer gradients ─────────────────────────────────────────
        # 🔨 TODO: Propagate the error back through W2 to the hidden layer.
        #          Multiply element-wise by the sigmoid derivative of hidden_output.
        delta_hidden = (delta_output @ ?) * sigmoid_derivative(?)
        dW1          = X.T @ ? / n
        db1          = np.sum(?, axis=0, keepdims=True) / n

        # ── Weight updates (gradient descent) ─────────────────────────────
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2

        # 🔨 TODO: Update W1 and b1 using the same pattern as W2 and b2 above.


SyntaxError: invalid syntax (1937632320.py, line 21)

---
### `train` and `predict` — Training loop and predictions

The training loop ties everything together.
For each **epoch** (one full pass through the dataset), we:
1. Call `forward` to get predictions
2. Calculate the MSE loss
3. Call `backward` to update weights

These two methods are provided in full — read them carefully.


In [14]:
    def train(self, X, y, epochs=2000, verbose=True):
        """
        Train the network by running forward + backward for each epoch.

        Parameters
        ----------
        X       : training input, shape (n_samples, input_size)
        y       : training labels, shape (n_samples, 1)
        epochs  : number of full passes through the data
        verbose : if True, print loss every 200 epochs
        """
        self.loss_history = []

        for epoch in range(epochs):
            # Forward pass — compute predictions
            output = self.forward(X)

            # Compute MSE loss: 0.5 * mean((prediction - true)^2)
            loss = 0.5 * np.mean((output - y) ** 2)
            self.loss_history.append(loss)

            # Backward pass — update weights
            self.backward(X, y, output)

            if verbose and (epoch + 1) % 200 == 0:
                print(f"Epoch {epoch + 1:>5} / {epochs}   Loss: {loss:.6f}")

    def predict(self, X, threshold=0.5):
        """Return binary class labels (0 or 1) for input X."""
        probabilities = self.forward(X)
        return (probabilities >= threshold).astype(int)


---
### Structure check ✅

Once you have filled in all four TODO gaps, run the cell below to verify your class is wired up correctly before moving to training.


In [15]:
# ── Run this cell to check your class structure ──────────────────────────────
np.random.seed(42)
nn_check = NeuralNetwork(input_size=2, hidden_size=4, output_size=1)

print("Checking attribute shapes...")
assert nn_check.W1.shape == (2, 4), f"W1 shape wrong: expected (2, 4), got {nn_check.W1.shape}"
assert nn_check.b1.shape == (1, 4), f"b1 shape wrong: expected (1, 4), got {nn_check.b1.shape}"
assert nn_check.W2.shape == (4, 1), f"W2 shape wrong: expected (4, 1), got {nn_check.W2.shape}"
assert nn_check.b2.shape == (1, 1), f"b2 shape wrong: expected (1, 1), got {nn_check.b2.shape}"
print("  W1:", nn_check.W1.shape, "✅")
print("  W2:", nn_check.W2.shape, "✅")

print("\nChecking forward pass...")
test_out = nn_check.forward(np.array([[0.5, -0.3]]))
assert test_out.shape == (1, 1), f"forward output shape wrong: {test_out.shape}"
assert 0 < float(test_out) < 1, "forward output should be between 0 and 1 (sigmoid output)"
print(f"  Output: {float(test_out):.4f}  (should be between 0 and 1) ✅")

print("\nAll structure checks passed! Proceed to Part 2.")


Checking attribute shapes...


AssertionError: W2 shape wrong: expected (4, 1), got (2, 4)

---
## Part 2 — Build and Train

With the class complete, we now load a dataset, train the model, and watch the loss decrease.

### The dataset — make_moons 🌙

`make_moons` generates two interleaved crescent shapes. It is a classic test for neural networks
because the two classes are **not linearly separable** — a straight line cannot divide them.
This means the hidden layer's non-linear transformation is essential for good accuracy.


In [ ]:
# ── Generate and split the dataset ───────────────────────────────────────────
X_raw, y_raw = make_moons(n_samples=500, noise=0.2, random_state=42)

# Reshape y to a column vector (n_samples, 1) — required by our network
y = y_raw.reshape(-1, 1)

# Standardise features (zero mean, unit variance) — helps gradient descent converge
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Input features   : {X_train.shape[1]}")


In [ ]:
# ── Visualise the dataset ─────────────────────────────────────────────────────
plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1],
            c=y_raw, cmap='coolwarm', edgecolors='k', linewidths=0.4, s=40)
plt.title("Dataset — make_moons")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()

print("Can you see why a straight line can't separate these two classes?")


### Create and train the network

The cell below creates a network with `hidden_size=8` and trains it for 2000 epochs.
Watch the loss decrease as you run it — each epoch the weights get a little better.


In [ ]:
# ── Instantiate and train ─────────────────────────────────────────────────────
np.random.seed(42)

nn = NeuralNetwork(
    input_size    = X_train.shape[1],   # 2 features
    hidden_size   = 8,
    output_size   = 1,
    learning_rate = 1.0
)

nn.train(X_train, y_train, epochs=2000, verbose=True)


In [ ]:
# ── Plot the training loss curve ──────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(nn.loss_history, color='steelblue', linewidth=1.5)
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Starting loss : {nn.loss_history[0]:.4f}")
print(f"Final loss    : {nn.loss_history[-1]:.4f}")


---
## Part 3 — Evaluate and Reflect

### Accuracy on training and test sets

A model that scores well on training data but poorly on test data is **overfitting** —
it has memorised training examples rather than learning general patterns.


In [ ]:
# ── Calculate accuracy ────────────────────────────────────────────────────────
def accuracy(model, X, y_true):
    predictions = model.predict(X)
    return np.mean(predictions == y_true) * 100

train_acc = accuracy(nn, X_train, y_train)
test_acc  = accuracy(nn, X_test,  y_test)

print(f"Training accuracy : {train_acc:.1f}%")
print(f"Test accuracy     : {test_acc:.1f}%")

gap = train_acc - test_acc
print(f"\nAccuracy gap      : {gap:.1f} percentage points")
if gap > 5:
    print("⚠️  Large gap — the model may be overfitting.")
else:
    print("✅  Small gap — the model is generalising well.")


### Decision boundary

In [ ]:
# ── Plot the decision boundary ────────────────────────────────────────────────
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1],
                c=y.ravel(), cmap='coolwarm', edgecolors='k', linewidths=0.4, s=30)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.tight_layout()
    plt.show()

plot_decision_boundary(nn, X_test, y_test,
                       title=f"Decision Boundary — Test Set  (accuracy: {test_acc:.1f}%)")


---
### Reflection questions

Answer the following questions in the comment blocks below.

**Q1.** Look at your loss curve. Does the loss decrease smoothly, or does it spike and fluctuate?
What does each pattern tell you about the learning rate?

**Q2.** What is the gap between your training and test accuracy?
Is there evidence of overfitting? How do you know?

**Q3.** Describe in your own words what backpropagation does and why it is necessary.
Your answer should mention gradients, weights, and the direction of information flow.


In [ ]:
# Q1 — Loss curve and learning rate:
#


In [ ]:
# Q2 — Overfitting assessment:
#


In [ ]:
# Q3 — Backpropagation in your own words:
#


---
## Extension — Experiment and Improve

Re-run the experiment cell below with different hyperparameter values.
Record your results in the table.

| `hidden_size` | `learning_rate` | Train acc | Test acc | Observation |
|--------------|----------------|-----------|----------|-------------|
| 8 | 1.0 | | | baseline |
| 2 | 1.0 | | | |
| 64 | 1.0 | | | |
| 8 | 0.01 | | | |
| 8 | 5.0 | | | |

**Questions to consider**
- Which `hidden_size` gives the best test accuracy? At what point does making it larger stop helping?
- What happens to the loss curve when `learning_rate` is very large? Very small?
- A `hidden_size` of 2 can only form 2 internal representations of the data. What does this tell you about the limits of small networks?


In [ ]:
# ── Experiment cell ───────────────────────────────────────────────────────────
np.random.seed(42)

nn_exp = NeuralNetwork(
    input_size    = X_train.shape[1],
    hidden_size   = 8,     # 🔨 change me
    output_size   = 1,
    learning_rate = 1.0    # 🔨 change me
)

nn_exp.train(X_train, y_train, epochs=2000, verbose=False)

train_acc_exp = accuracy(nn_exp, X_train, y_train)
test_acc_exp  = accuracy(nn_exp, X_test,  y_test)
print(f"Train: {train_acc_exp:.1f}%   Test: {test_acc_exp:.1f}%")

# Loss curve
plt.figure(figsize=(8, 3))
plt.plot(nn_exp.loss_history, color='coral', linewidth=1.5)
plt.title(f"Experiment Loss  (hidden={nn_exp.hidden_size}, lr={nn_exp.learning_rate})")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
